# Phase B v4: zero8 복구분 FinBERT (en / ja / de / es)

**목적**: scored=0 8개 기업 (Honda/Ford/Glencore/Albemarle/Lucid/BASF/Asahi Kasei/SQM)의
Google News RSS로 복구한 title-bearing 데이터에 대해 FinBERT 추론.

**입력** (로컬 `scripts/extract_zero8_finbert_input.py` 로 생성)
- `data/processed/finbert_input_zero8_en.parquet`
- `data/processed/finbert_input_zero8_ja.parquet`
- `data/processed/finbert_input_zero8_de.parquet`
- `data/processed/finbert_input_zero8_es.parquet`

**출력** (Colab)
- `data/processed/finbert_results_zero8_en.parquet`
- `data/processed/finbert_results_zero8_ja.parquet`
- `data/processed/finbert_results_zero8_de.parquet`
- `data/processed/finbert_results_zero8_es.parquet`

**사용 모델** (HF Hub 검증됨, 2026-04)
| 언어 | 모델 | 근거 |
|---|---|---|
| en | `ProsusAI/finbert` | 우리 gold-standard, 84.2M DL |
| ja | `christian-phu/bert-finetuned-japanese-sentiment` | 금융 뉴스 도메인, 763K DL |
| de | `cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual` | multi-lingual fallback, 10.6M DL |
| es | `cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual` | 동일 모델 |

**제약**: de/es는 twitter 도메인 모델이라 금융 특화가 아님. 해당 언어 비율이 낮다면 수용 가능.

**주의**: 모델별 `id2label` 순서가 다르므로 동적 매핑(`find_label_indices`) 사용.

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT_DIR = '/content/drive/MyDrive/나비효과'  # ← 본인 경로로 수정
os.chdir(PROJECT_DIR)
print('cwd:', os.getcwd())

In [ ]:
!pip install -q transformers torch pandas pyarrow tqdm sentencepiece fugashi unidic-lite
import torch
print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. 공용 배치 추론 함수

v3와 동일: `id2label`을 동적으로 읽어 pos/neg/neu 인덱스를 자동 추출.

In [ ]:
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def find_label_indices(id2label):
    """id2label에서 pos/neg/neu 인덱스 동적 추출. 다양한 표기/언어 허용."""
    mapping = {i: str(v).lower().strip() for i, v in id2label.items()}
    pos_keywords = {'positive', 'pos', 'bullish', 'good', 'label_2', '正面', '긍정', 'ポジティブ'}
    neg_keywords = {'negative', 'neg', 'bearish', 'bad',  'label_0', '负面', '부정', 'ネガティブ'}
    neu_keywords = {'neutral',  'neu', 'flat', 'label_1', '中性', '중립', 'ニュートラル'}
    pi = ni = ui = None
    for i, v in mapping.items():
        if any(k in v for k in pos_keywords) and pi is None: pi = i
        elif any(k in v for k in neg_keywords) and ni is None: ni = i
        elif any(k in v for k in neu_keywords) and ui is None: ui = i
    return pi, ni, ui, mapping

def run_finbert(model_name, input_path, output_path, batch_size=64, max_len=256):
    print(f'\n{"="*70}\n[{model_name}]\n  → {input_path}\n{"="*70}')
    if not os.path.exists(input_path):
        print(f'  ⚠️  input not found, skipping: {input_path}')
        return None
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name).eval().to(device)
    pi, ni, ui, mapping = find_label_indices(model.config.id2label)
    print(f'  id2label   : {mapping}')
    print(f'  pos_idx={pi}, neg_idx={ni}, neu_idx={ui}')
    if pi is None or ni is None:
        raise RuntimeError(f'Could not infer pos/neg indices from id2label={mapping}')

    df = pd.read_parquet(input_path)
    print(f'  input rows : {len(df):,}')
    if len(df) == 0:
        print('  (empty input, skipping)')
        return None

    texts = df['title'].astype(str).tolist()
    n = len(texts)
    neg = np.zeros(n, dtype=np.float32)
    neu = np.zeros(n, dtype=np.float32)
    pos = np.zeros(n, dtype=np.float32)

    for i in tqdm(range(0, n, batch_size), desc=model_name.split('/')[-1]):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=max_len, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        pos[i:i+len(batch)] = probs[:, pi]
        neg[i:i+len(batch)] = probs[:, ni]
        if ui is not None:
            neu[i:i+len(batch)] = probs[:, ui]
        else:
            neu[i:i+len(batch)] = 1.0 - probs[:, pi] - probs[:, ni]

    df['finbert_neg']   = neg
    df['finbert_neu']   = neu
    df['finbert_pos']   = pos
    df['finbert_score'] = (pos - neg).astype(np.float32)
    df['finbert_model'] = model_name

    print('\n  distribution:')
    print(df[['finbert_neg','finbert_neu','finbert_pos','finbert_score']].describe().round(4))
    strong = int((df['finbert_score'].abs() > 0.3).sum())
    print(f'  strong signal (|score|>0.3): {strong:,} ({100*strong/len(df):.1f}%)')

    df_out = df[['rep_event_id','title','finbert_neg','finbert_neu','finbert_pos','finbert_score','finbert_model']]
    df_out.to_parquet(output_path, index=False, compression='zstd')
    print(f'  saved      : {output_path}')

    del model; torch.cuda.empty_cache()
    return df_out

## 2. 영어: ProsusAI/finbert (gold standard)

In [ ]:
df_en = run_finbert(
    model_name = 'ProsusAI/finbert',
    input_path = 'data/processed/finbert_input_zero8_en.parquet',
    output_path= 'data/processed/finbert_results_zero8_en.parquet',
    batch_size = 64
)

In [ ]:
if df_en is not None:
    print('=== EN most NEGATIVE ===')
    for _, r in df_en.nsmallest(5, 'finbert_score').iterrows():
        print(f'  {r.finbert_score:+.3f} | {r.title[:90]}')
    print('\n=== EN most POSITIVE ===')
    for _, r in df_en.nlargest(5, 'finbert_score').iterrows():
        print(f'  {r.finbert_score:+.3f} | {r.title[:90]}')

## 3. 일본어: christian-phu/bert-finetuned-japanese-sentiment

In [ ]:
df_ja = run_finbert(
    model_name = 'christian-phu/bert-finetuned-japanese-sentiment',
    input_path = 'data/processed/finbert_input_zero8_ja.parquet',
    output_path= 'data/processed/finbert_results_zero8_ja.parquet',
    batch_size = 64
)

In [ ]:
if df_ja is not None:
    print('=== JA most NEGATIVE ===')
    for _, r in df_ja.nsmallest(5, 'finbert_score').iterrows():
        print(f'  {r.finbert_score:+.3f} | {r.title[:90]}')
    print('\n=== JA most POSITIVE ===')
    for _, r in df_ja.nlargest(5, 'finbert_score').iterrows():
        print(f'  {r.finbert_score:+.3f} | {r.title[:90]}')

## 4. 독일어: cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual

⚠️ **제약**: Twitter 도메인 모델. 금융 뉴스에 최적화되지 않음.
de/es는 대상 기업이 적어 커버리지가 크지 않을 것으로 예상.

In [ ]:
df_de = run_finbert(
    model_name = 'cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual',
    input_path = 'data/processed/finbert_input_zero8_de.parquet',
    output_path= 'data/processed/finbert_results_zero8_de.parquet',
    batch_size = 64
)

## 5. 스페인어: 동일 multilingual 모델

In [ ]:
df_es = run_finbert(
    model_name = 'cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual',
    input_path = 'data/processed/finbert_input_zero8_es.parquet',
    output_path= 'data/processed/finbert_results_zero8_es.parquet',
    batch_size = 64
)

## 6. 요약

In [ ]:
print('=' * 60)
print('zero8 FinBERT 완료 요약')
print('=' * 60)
for lang, df in [('en', df_en), ('ja', df_ja), ('de', df_de), ('es', df_es)]:
    if df is None:
        print(f'  {lang}: (skipped)')
        continue
    mean_s = df['finbert_score'].mean()
    std_s = df['finbert_score'].std()
    strong = int((df['finbert_score'].abs() > 0.3).sum())
    print(f'  {lang}: {len(df):>6,}행  mean={mean_s:+.3f}  std={std_s:.3f}  strong={strong:,} ({100*strong/len(df):.1f}%)')

print('\n다음 단계 (로컬):')
print('  1. 4개 finbert_results_zero8_*.parquet 동기화')
print('  2. python scripts/merge_zero8_finbert.py   # v7 FinBERT 컬럼 채우기')
print('  3. python scripts/diagnose_v7_bias.py      # 편중 재진단')